In [1]:
from pathlib import Path

In [2]:
from torchvision import transforms
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import torch

In [3]:
device = 'cuda' if torch.accelerator.is_available() else 'cpu'

### Data Prep

In [4]:
data_dir = Path('data/fashion')
train_data_dir = data_dir / 'train'
test_data_dir = data_dir / 'test'

In [28]:
transform = transforms.Compose(
    [transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))]
)

In [29]:
should_download = !Path.exists(data_dir)

train_data_full = FashionMNIST(train_data_dir, transform=transform, download=should_download)
test_data = FashionMNIST(test_data_dir, transform=transform, download=should_download)

In [30]:
train_size = int(len(train_data_full) * 0.9)
val_size = len(train_data_full) - train_size
train_data, val_data = random_split(train_data_full, lengths=[train_size, val_size])

In [31]:
train_loader = DataLoader(train_data, batch_size=4, num_workers=2, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=4, num_workers=2, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=4, num_workers=2, shuffle=True, pin_memory=True)

### Model

In [36]:
class CNN(nn.Module):
    
    def __init__(self, in_layers=1, h1=6, h2=16, out_layers=10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_layers, h1, 3)
        self.conv2 = nn.Conv2d(h1, h2, 3)

        self.classifier = nn.Sequential(
            nn.Linear(h2 * 5 * 5, out_layers)
        )

    def forward(self, x):
        x = F.max_pool2d(self.conv1(x), 2)
        x = F.relu(x)
        x = F.max_pool2d(self.conv2(x), 2)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [37]:
model = CNN()

### Training

In [44]:
epochs = 200
criterion = nn.CrossEntropyLoss()
optimizer = Adam(params=model.parameters())

losses = []

In [45]:
model.to(device)
model.train()
for epoch in range(epochs):
    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)
        
        optimizer.zero_grad()
        
        y_pred = model(image)
        y_pred.shape # should be [4, 10]
        loss = criterion(y_pred, label)
        losses.append(loss.item())
        
        loss.backward()
        optimizer.step()

    if epoch % 5 == 0:
        print(f'Epoch: {epoch}/{epochs}, Loss: {loss.item()}')

/home/jamnic/miniconda3/envs/pytorch-gen-ai/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


KeyboardInterrupt: 